# Get citing & cited opinions metadata for Bankruptcy Courts

In the previous experiments, we've primarily focused on SCOTUS. This is to expand the dataset to bankruptcy courts in the Federal jurisdiction, including the Bankruptcy appeals courts and the Bankruptcy panel courts. 

This notebook documents the steps I undertook to:
1. Identify the Federal bankruptcy courts to include
2. Use Django Shell to sample target cases in the target courts from CL Replica
3. Use Django Shell to get all cases that cited the sampled target cases - these are the citing cases
4. Use Django Shell to get all authorities for the citing cases - these are the cited cases, including ones in the specified courts (target cases) and the ones not in the specified courts
5. Use Django Shell to get related metadata for the citing and cited cases

# Import Libaries

In [1]:
import json

import numpy as np
import pandas as pd

# Load Court Hierarchy data

In [2]:
df = pd.read_csv("../experiments_624/court_hierarchy_flp.csv")
df.head()

,id,full_name,jurisdiction,jurisdiction_name,jurisdiction_type,jurisdiction_state,appeals_to_full_name,appeals_to_id,note
0,scotus,Supreme Court of the United States,F,Federal Appellate,Federal,Federal,NaN,NaN,NaN
1,cafc,Court of Appeals for the Federal Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN
2,ca1,Court of Appeals for the First Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN
3,ca2,Court of Appeals for the Second Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN
4,ca3,Court of Appeals for the Third Circuit,F,Federal Appellate,Federal,Federal,Supreme Court of the United States,scotus,NaN


In [3]:
df["jurisdiction_name"].value_counts()

jurisdiction_name
Federal Bankruptcy          141
Federal District             94
State Appellate              57
State Supreme                52
Federal Appellate            14
Federal Bankruptcy Panel      8
State Trial                   6
Territory Supreme             5
Territory Trial               4
Territory Appellate           2
Name: count, dtype: int64

In [4]:
fed_bankruptcy = df[df["jurisdiction_name"] == "Federal Bankruptcy"]
fed_bankruptcy["full_name"]

20          United States Bankruptcy Court, M.D. Alabama
21          United States Bankruptcy Court, N.D. Alabama
22          United States Bankruptcy Court, S.D. Alabama
23             United States Bankruptcy Court, D. Alaska
24             United States Bankruptcy Court, D. Alaska
                             ...                        
158    United States Bankruptcy Court, Northern Maria...
159      United States Bankruptcy Court, D. Rhode Island
160       United States Bankruptcy Court, D. Puerto Rico
161       United States Bankruptcy Court, D. Puerto Rico
162    United States Bankruptcy Court, D. Virgin Islands
Name: full_name, Length: 141, dtype: object

In [5]:
fed_bankruptcy_panel = df[df["jurisdiction_name"] == "Federal Bankruptcy Panel"]
fed_bankruptcy_panel["full_name"]

14      Bankruptcy Appellate Panel of the First Circuit
15     Bankruptcy Appellate Panel of the Second Circuit
16      Bankruptcy Appellate Panel of the Sixth Circuit
17    United States Bankruptcy Appellate Panel for t...
18    United States Bankruptcy Appellate Panel for t...
19      Bankruptcy Appellate Panel of the Tenth Circuit
72                 Bankruptcy Appellate Panel, D. Maine
76          Bankruptcy Appellate Panel of Massachusetts
Name: full_name, dtype: object

In [6]:
fed_bankruptcy["jurisdiction_state"].value_counts()

jurisdiction_state
California                  8
Oklahoma                    6
Tennessee                   6
Iowa                        4
New York                    4
Ohio                        4
Arkansas                    4
Missouri                    4
Michigan                    4
Texas                       4
Washington                  4
Kentucky                    4
Pennsylvania                3
Louisiana                   3
Alabama                     3
Illinois                    3
Georgia                     3
Florida                     3
North Carolina              3
Virginia                    2
Utah                        2
West Virginia               2
South Dakota                2
Wisconsin                   2
Rhode Island                2
Wyoming                     2
Guam                        2
Oregon                      2
Northern Mariana Islands    2
Puerto Rico                 2
North Dakota                2
Nebraska                    2
New Mexico           

In [7]:
court_ids = fed_bankruptcy["id"].to_list() + fed_bankruptcy_panel["id"].to_list()
len(court_ids)

149

In [8]:
court_ids

['almb',
 'alnb',
 'alsb',
 'akb',
 'akb',
 'arb',
 'arb',
 'areb',
 'areb',
 'arwb',
 'arwb',
 'cacb',
 'cacb',
 'caeb',
 'caeb',
 'canb',
 'canb',
 'casb',
 'casb',
 'cob',
 'cob',
 'ctb',
 'deb',
 'dcb',
 'flmb',
 'flnb',
 'flsb',
 'gamb',
 'ganb',
 'gasb',
 'hib',
 'hib',
 'idb',
 'idb',
 'ilcb',
 'ilnb',
 'ilsb',
 'innb',
 'insb',
 'ianb',
 'ianb',
 'iasb',
 'iasb',
 'ksb',
 'ksb',
 'kyeb',
 'kyeb',
 'kywb',
 'kywb',
 'laeb',
 'lamb',
 'lawb',
 'meb',
 'meb',
 'mdb',
 'mab',
 'mab',
 'mieb',
 'mieb',
 'miwb',
 'miwb',
 'mnb',
 'mnb',
 'msnb',
 'mssb',
 'moeb',
 'moeb',
 'mowb',
 'mowb',
 'mtb',
 'mtb',
 'nebraskab',
 'nebraskab',
 'nvb',
 'nvb',
 'nhb',
 'njb',
 'nmb',
 'nmb',
 'nyeb',
 'nynb',
 'nysb',
 'nywb',
 'nceb',
 'ncmb',
 'ncwb',
 'ndb',
 'ndb',
 'ohnb',
 'ohnb',
 'ohsb',
 'ohsb',
 'okeb',
 'okeb',
 'oknb',
 'oknb',
 'okwb',
 'okwb',
 'orb',
 'orb',
 'paeb',
 'pamb',
 'pawb',
 'nhb',
 'rib',
 'scb',
 'sdb',
 'sdb',
 'tneb',
 'tneb',
 'tnmb',
 'tnmb',
 'tnwb',
 'tnwb',
 't

# Use Django Shell to get the list of target case cluster ids and citing case cluster ids

In [9]:
with open("data/citing_ids.txt", "r") as f:
    citing_ids = [line.strip() for line in f]
len(citing_ids)

746

In [10]:
with open("data/target_ids.txt", "r") as f:
    target_ids = [int(line.strip()) for line in f]
len(target_ids)

196

## Ensure the citing cases are not already part of the SCOTUS set in review

In [11]:
scotus = pd.read_json("../experiments_624/data/scotus_citing_cited_sampled.json")
len(scotus)

21925

In [12]:
scotus_citing = list(set(scotus["citing_cluster_id"].to_list()))
len(scotus_citing)

482

In [13]:
assert len(set(citing_ids) - set(scotus_citing)) == len(citing_ids)

# Use Django Shell to get the citing cases metadata

In [14]:
with open('data/citing_opinions.json', 'r') as f:
    results = json.load(f)

In [15]:
records = []
for case_id, case_data in results.items():
    record = {'citing_cluster_id': int(case_id)}
    record.update({k: v for k, v in case_data.items()})
    
    opinion_filenames = [op['opinion_filename'] for op in case_data.get('opinion_data', [])]
    record['opinion_filenames'] = opinion_filenames
    
    records.append(record)

citing_df = pd.DataFrame(records)
citing_df.head()

,citing_cluster_id,citing_url,citing_court_id,citing_court_name,opinion_data,cited_cluster_ids,opinion_filenames
0,878680,https://www.courtlistener.com/opinion/878680/i...,mont,Montana Supreme Court,"[{'opinion_id': 878680, 'opinion_api': None, '...","[876178, 876307, 876421, 876805, 876872, 87715...",[878680_010combined.txt]
1,886881,https://www.courtlistener.com/opinion/886881/w...,mont,Montana Supreme Court,"[{'opinion_id': 886881, 'opinion_api': None, '...","[885011, 885409, 886169, 886227, 886625, 35638...",[886881_010combined.txt]
2,886985,https://www.courtlistener.com/opinion/886985/g...,mont,Montana Supreme Court,"[{'opinion_id': 886985, 'opinion_api': None, '...","[880827, 881860, 882145, 882240, 883139, 88355...",[886985_010combined.txt]
3,887009,https://www.courtlistener.com/opinion/887009/j...,mont,Montana Supreme Court,"[{'opinion_id': 887009, 'opinion_api': None, '...","[877836, 878573, 880466, 882097, 884719, 88501...",[887009_010combined.txt]
4,1108230,https://www.courtlistener.com/opinion/1108230/...,lactapp,Louisiana Court of Appeal,"[{'opinion_id': 1108230, 'opinion_api': None, ...","[1097321, 1098848, 1134126, 1700159, 1724577, ...",[1108230_010combined.txt]


## Get all cited case ids to a list for extracting the metadata

In [16]:
#with open('data/cited_ids.txt', 'w') as f:
#    for each in list(set(citing_df["cited_cluster_ids"].explode().dropna().tolist())):
#        f.write(f"{each}\n")

# Use Django Shell to extract the metadata for the cited cases

In [17]:
with open('data/cited_opinions.json', 'r') as f:
    results = json.load(f)

In [18]:
cited_metadata = pd.DataFrame.from_dict(results, orient='index')
cited_metadata = cited_metadata.reset_index().rename(columns={'index': 'cited_cluster_id'})
cited_metadata["cited_cluster_id"] = cited_metadata["cited_cluster_id"].astype(int)
cited_metadata.head()

,cited_cluster_id,cited_url,cited_court_id,cited_court_name,cited_case_name_short,cited_case_name,cited_case_name_full,cited_citations
0,93,https://www.courtlistener.com/opinion/93/primi...,ca9,Court of Appeals for the Ninth Circuit,Primiano,Primiano v. Cook,"Marylou PRIMIANO; Charles Primiano, Plaintiffs...","[2010 WL 788906, 2010 U.S. App. LEXIS 5014, 81..."
1,98330,https://www.courtlistener.com/opinion/98330/he...,scotus,Supreme Court of the United States,Hendrick,Hendrick v. Maryland,Hendrick v. State of Maryland,"[1915 U.S. LEXIS 1848, 59 L. Ed. 385, 35 S. Ct..."
2,98427,https://www.courtlistener.com/opinion/98427/ma...,scotus,Supreme Court of the United States,Malloy,Malloy v. South Carolina,Malloy v. State of South Carolina,"[1915 U.S. LEXIS 1324, 59 L. Ed. 905, 35 S. Ct..."
3,163939,https://www.courtlistener.com/opinion/163939/g...,ca10,Court of Appeals for the Tenth Circuit,Goebel,Goebel v. Denver & Rio Grande Western Railroad,"Richard W. GOEBEL, Plaintiff-Appellee, v. the ...","[2003 WL 22311330, 2003 U.S. App. LEXIS 20702,..."
4,262262,https://www.courtlistener.com/opinion/262262/s...,ca8,Court of Appeals for the Eighth Circuit,,State Farm Mutual Automobile Insurance Company...,STATE FARM MUTUAL AUTOMOBILE INSURANCE COMPANY...,"[1963 U.S. App. LEXIS 3672, 324 F.2d 340]"


In [19]:
len(cited_metadata)

9912

# Create result_df by merging citing and cited metadatas

In [20]:
result_df = citing_df.explode("cited_cluster_ids").reset_index(drop=True)
len(result_df)

13248

In [21]:
result_df = result_df.rename(columns={"cited_cluster_ids": "cited_cluster_id"})
result_df.head()

,citing_cluster_id,citing_url,citing_court_id,citing_court_name,opinion_data,cited_cluster_id,opinion_filenames
0,878680,https://www.courtlistener.com/opinion/878680/i...,mont,Montana Supreme Court,"[{'opinion_id': 878680, 'opinion_api': None, '...",876178,[878680_010combined.txt]
1,878680,https://www.courtlistener.com/opinion/878680/i...,mont,Montana Supreme Court,"[{'opinion_id': 878680, 'opinion_api': None, '...",876307,[878680_010combined.txt]
2,878680,https://www.courtlistener.com/opinion/878680/i...,mont,Montana Supreme Court,"[{'opinion_id': 878680, 'opinion_api': None, '...",876421,[878680_010combined.txt]
3,878680,https://www.courtlistener.com/opinion/878680/i...,mont,Montana Supreme Court,"[{'opinion_id': 878680, 'opinion_api': None, '...",876805,[878680_010combined.txt]
4,878680,https://www.courtlistener.com/opinion/878680/i...,mont,Montana Supreme Court,"[{'opinion_id': 878680, 'opinion_api': None, '...",876872,[878680_010combined.txt]


In [22]:
result_df = result_df.merge(cited_metadata, how="left", on="cited_cluster_id")
len(result_df)

13248

In [23]:
result_df.columns

Index(['citing_cluster_id', 'citing_url', 'citing_court_id',
       'citing_court_name', 'opinion_data', 'cited_cluster_id',
       'opinion_filenames', 'cited_url', 'cited_court_id', 'cited_court_name',
       'cited_case_name_short', 'cited_case_name', 'cited_case_name_full',
       'cited_citations'],
      dtype='object')

In [24]:
result_df = result_df[['citing_cluster_id', 'citing_url', 'citing_court_id',
       'citing_court_name', 'opinion_data', 'opinion_filenames', 
       'cited_cluster_id', 'cited_url', 'cited_court_id', 'cited_court_name',
       'cited_case_name_short', 'cited_case_name', 'cited_case_name_full',
       'cited_citations']]

In [25]:
result_df.head()

,citing_cluster_id,citing_url,citing_court_id,citing_court_name,opinion_data,opinion_filenames,cited_cluster_id,cited_url,cited_court_id,cited_court_name,cited_case_name_short,cited_case_name,cited_case_name_full,cited_citations
0,878680,https://www.courtlistener.com/opinion/878680/i...,mont,Montana Supreme Court,"[{'opinion_id': 878680, 'opinion_api': None, '...",[878680_010combined.txt],876178,https://www.courtlistener.com/opinion/876178/e...,mont,Montana Supreme Court,Eschenburg,Eschenburg v. Eschenburg,"BETTY GUNN ESCHENBURG, Plaintiff and Responden...","[1976 Mont. LEXIS 542, 171 Mont. 247, 557 P.2d..."
1,878680,https://www.courtlistener.com/opinion/878680/i...,mont,Montana Supreme Court,"[{'opinion_id': 878680, 'opinion_api': None, '...",[878680_010combined.txt],876307,https://www.courtlistener.com/opinion/876307/e...,mont,Montana Supreme Court,Englund,Englund v. Englund,"DANNIE ENGLUND, Plaintiff and Respondent, v. C...","[169 Mont. 418, 547 P.2d 841]"
2,878680,https://www.courtlistener.com/opinion/878680/i...,mont,Montana Supreme Court,"[{'opinion_id': 878680, 'opinion_api': None, '...",[878680_010combined.txt],876421,https://www.courtlistener.com/opinion/876421/i...,mont,Montana Supreme Court,In Re Gore,In Re Gore,"In Re JERRY WAYNE GORE, RHONDA GAIL GORE, and ...","[1977 Mont. LEXIS 604, 174 Mont. 321, 570 P.2d..."
3,878680,https://www.courtlistener.com/opinion/878680/i...,mont,Montana Supreme Court,"[{'opinion_id': 878680, 'opinion_api': None, '...",[878680_010combined.txt],876805,https://www.courtlistener.com/opinion/876805/o...,mont,Montana Supreme Court,O'Neill,O'Neill v. O'Neill,"In Re the MARRIAGE of THERESA C. O’NEILL, Peti...","[1979 Mont. LEXIS 943, 184 Mont. 415, 603 P.2d..."
4,878680,https://www.courtlistener.com/opinion/878680/i...,mont,Montana Supreme Court,"[{'opinion_id': 878680, 'opinion_api': None, '...",[878680_010combined.txt],876872,https://www.courtlistener.com/opinion/876872/b...,mont,Montana Supreme Court,Byrd,Byrd v. Columbia Falls Lions Club,"HERMAN BYRD, D/B/A/ BYRD’S FOOD MART, Plaintif...","[1979 Mont. LEXIS 880, 183 Mont. 330, 599 P.2d..."


## Tag the target cases from the target courts

In [26]:
result_df.loc[result_df["cited_cluster_id"].isin(target_ids), "cited_target"] = 1
result_df.loc[~result_df["cited_cluster_id"].isin(target_ids), "cited_target"] = 0

## Do some EDA

In [27]:
eda_cols = ['citing_cluster_id', 'citing_court_name', 'cited_cluster_id', 'cited_court_name', 'cited_target']

for col in eda_cols:
    print("----------")
    print(result_df[col].nunique())
    display(result_df[col].value_counts())

----------
746


citing_cluster_id
4291288     826
2314561     139
10305153    131
442776      128
1658111     119
           ... 
6201397       1
5660273       1
7613147       1
9400572       1
1498222       1
Name: count, Length: 746, dtype: int64

----------
110


citing_court_name
Court of Appeals of Texas                                           1514
Appellate Division of the Supreme Court of the State of New York     986
Michigan Court of Appeals                                            637
Supreme Court of Alabama                                             583
Montana Supreme Court                                                575
                                                                    ... 
United States Bankruptcy Court, N.D. Alabama                           8
Supreme Court of Oklahoma                                              4
District Court, W.D. Pennsylvania                                      4
The Superior Court of the City of New York and Buffalo                 2
The Superior Court of New York City                                    2
Name: count, Length: 110, dtype: int64

----------
9912


cited_cluster_id
5682590    124
1878219     55
1789533     50
7959560     34
885011      31
          ... 
725022       1
724912       1
723835       1
714170       1
7893260      1
Name: count, Length: 9912, dtype: int64

----------
204


cited_court_name
New York Court of Appeals                                           867
Appellate Division of the Supreme Court of the State of New York    749
Supreme Court of Alabama                                            685
Court of Appeals of Texas                                           681
Supreme Court of the United States                                  616
                                                                   ... 
District Court, S.D. Alabama                                          1
District Court, D. Puerto Rico                                        1
District Court, W.D. Arkansas                                         1
District Court, S.D. California                                       1
Court of Appeals of South Carolina                                    1
Name: count, Length: 204, dtype: int64

----------
2


cited_target
0.0    12502
1.0      746
Name: count, dtype: int64

In [28]:
df_target = result_df[result_df["cited_target"] == 1]
print(df_target["cited_court_name"].nunique())
df_target["cited_court_name"].value_counts()

44


cited_court_name
New York Court of Appeals                                           124
Supreme Court of Alabama                                             96
Appellate Division of the Supreme Court of the State of New York     67
Michigan Supreme Court                                               56
Montana Supreme Court                                                35
Supreme Court of Kansas                                              34
Court of Appeals of Texas                                            33
Louisiana Court of Appeal                                            27
Supreme Court of Florida                                             26
District Court of Appeal of Florida                                  21
Supreme Court of Missouri                                            17
Supreme Court of New Hampshire                                       17
Court of Appeals of Wisconsin                                        17
Appellate Court of Illinois                    

# Save the data for future use

In [28]:
result_df.to_json("data/fed_citing_cited.json")